# Identifiability of route usage

A standalone notebook for the architecture section. It runs on its own.

## The claim this section makes, and the one it does not

It does **not** claim that an additive two-route readout is a new architecture.
It is not: a transformer residual stream already combines an attention read
additively with everything else, and additive attention readouts long predate
this work. Nor does it claim higher accuracy — every structure here reaches
ceiling, and the accuracy column exists only to show that it cannot separate
them.

What it claims is narrower and testable:

> Measuring which algorithm a hybrid network is running requires a readout in
> which the two routes are separable. Standard hybrid readouts do not have this
> property. We give the design conditions under which the measurement is
> unbiased, and verify them against ground truth.

That makes the architecture an **instrument** supporting the paper's finding,
rather than a second contribution competing with it. It is a smaller claim and
a much harder one to reject.

## What is measured

Five readouts on an identical, capacity-matched GRU encoder:

| | keys/values | combination |
|---|---|---|
| `additive` (ours) | exclude $h_T$ | $W_h f(h_T) + W_c g(c)$ |
| `additive_incl` | **include** $h_T$ | $W_h f(h_T) + W_c g(c)$ |
| `shared_proj` | exclude $h_T$ | $W(f(h_T) + g(c))$ |
| `concat_mlp` | exclude $h_T$ | $W(\mathrm{MLP}[f(h_T); g(c)])$ |
| `gated` | exclude $h_T$ | $W(s f(h_T) + (1-s) g(c))$ |

Two failure modes, two calibrations, both with real ground truth.

**Cross-talk.** Train with one branch resampled at every step, so it carries no
trial-specific information and the network provably cannot use it. Then measure
normally. Any dependence reported on that branch is a false positive, and its
size is the number that goes in the table where a checkmark would otherwise go.
This catches readouts that entangle the branches through a nonlinear mixer.

**State leakage.** Cross-talk does not catch the other failure: with $h_T$ among
the values, the retrieval route can carry a copy of the recurrent state, so a
model running recurrence reports high attention dependence. The diagnostic
splits the read into the final-timestep slot and everything earlier and ablates
each alone. Dependence concentrated in the slot is not retrieval.

Both are backed by a third, structure-independent ground truth: decodability of
the answer from each branch, measured with a fresh probe on a frozen encoder.

## The result that protects the paper

The last block applies the same price to all five structures. If the drift
happens everywhere but is only *readable* in one, the finding is
architecture-general and the readout is an instrument — which is a stronger
position than "this only happens in our model".

## Running it

About 350k optimiser steps, well under an hour on a GPU. `PRESET = "smoke"`
first: every block, real grids, tiny budgets, a few minutes. Runs resume.


In [ ]:
# !pip install torch numpy pandas matplotlib

import copy, glob, hashlib, itertools, json, math, os, pathlib, platform, random, time
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================== CHOOSE HERE ==============================
PRESET  = "full"     # "smoke" (~5 min pipeline check) | "quick" | "full"
RESUME  = True
WORKDIR = os.environ.get("PAPER_WORKDIR", "./identifiability")
# =========================================================================

RUN = {"sufficiency": True, "redundancy": True, "identify": True,
       "table": True, "figures": True}


def S(full, quick=None, smoke=None):
    return {"full": full,
            "quick": quick if quick is not None else max(full // 3, 100),
            "smoke": smoke if smoke is not None else max(full // 25, 80)}[PRESET]


SEEDS = {"full": [0, 1, 2, 3, 4], "quick": [0, 1], "smoke": [0]}[PRESET]
LOAD  = 8
STEPS = S(2500)          # every model is trained with the same budget
PROBE_STEPS = S(600)     # readout probes on frozen representations
LOG_EVERY = max(S(250), 20)
LAM = 0.10               # the price, from the main paper's calibration

_sfx = "" if PRESET == "full" else f"_{PRESET}"
WORK = pathlib.Path(WORKDIR).expanduser().resolve()
RESDIR, CKPTDIR = WORK / f"results{_sfx}", WORK / f"checkpoints{_sfx}"
FIGDIR, TABDIR = WORK / f"figures{_sfx}", WORK / f"tables{_sfx}"
for _d in (RESDIR, CKPTDIR, FIGDIR, TABDIR):
    _d.mkdir(parents=True, exist_ok=True)
TAG = "ident"

print(f"device {DEVICE} | preset {PRESET} | resume {RESUME}")
print(f"work dir {WORK} | seeds {SEEDS} | {STEPS} steps per model")
if PRESET == "smoke":
    print("!! SMOKE preset: numbers are meaningless, this only checks the pipeline.")


def csv(name):
    return str(RESDIR / f"{name}__{TAG}.csv")


def set_seed(s):
    torch.manual_seed(s); np.random.seed(s); random.seed(s)


## Persistence and resume

Every unit of work is content-hashed on its specification and written to disk
the moment it finishes, so restarting runs only what is missing. Delete a
block's `.jsonl` to force recomputation.


In [ ]:
def _jsonable(o):
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, torch.Tensor):
        return o.detach().cpu().tolist()
    if isinstance(o, pathlib.Path):
        return str(o)
    return str(o)


def keyof(spec):
    """Content hash of a run specification. Two runs with the same spec are the
    same experiment, so the second one is skipped."""
    return hashlib.md5(
        json.dumps(spec, sort_keys=True, default=_jsonable).encode()).hexdigest()[:16]


def cfg_tag(cfg=None):
    if cfg is None:
        return "default"
    return ",".join(f"{k}={getattr(cfg, k)}" for k in sorted(cfg.__dataclass_fields__))


class Store:
    """Append-only JSONL of finished units of work, one file per block.

    A unit is whatever a block can afford to lose: usually one (condition, seed)
    training run. Records are flushed immediately, so a killed kernel loses at
    most the run in flight. A truncated final line is discarded on load.
    """

    def __init__(self, block):
        self.block = block
        self.path = RESDIR / f"{block}__{TAG}.jsonl"
        self.recs, bad = {}, 0
        if RESUME and self.path.exists():
            for line in self.path.read_text().splitlines():
                if not line.strip():
                    continue
                try:
                    r = json.loads(line)
                    self.recs[r["_key"]] = r
                except (json.JSONDecodeError, KeyError):
                    bad += 1
            if bad:
                print(f"  [{block}] discarded {bad} unreadable record(s)")

    def has(self, spec):
        return RESUME and keyof(spec) in self.recs

    def get(self, spec):
        return self.recs[keyof(spec)]

    def put(self, spec, payload):
        r = {"_key": keyof(spec), "_spec": spec, "_t": time.time(), **payload}
        self.recs[r["_key"]] = r
        with open(self.path, "a") as f:
            f.write(json.dumps(r, default=_jsonable) + "\n")
            f.flush(); os.fsync(f.fileno())
        return r

    def frame(self):
        """Flatten to one row per result. A record either carries metric fields
        directly or a `rows` list, in which case each entry becomes a row."""
        out = []
        for r in self.recs.values():
            spec = {k: v for k, v in r["_spec"].items()}
            body = {k: v for k, v in r.items()
                    if not k.startswith("_") and k not in ("trace", "rows")}
            if "rows" in r:
                for row in r["rows"]:
                    out.append({**spec, **body, **row})
            else:
                out.append({**spec, **body})
        return pd.DataFrame(out)

    def traces(self):
        out = []
        for r in self.recs.values():
            for t in r.get("trace", []):
                out.append({**r["_spec"], **t})
        return pd.DataFrame(out)

    def elapsed(self):
        return sum(r.get("_elapsed", 0.0) for r in self.recs.values())


def cached(store, spec, fn, label=None):
    """Run `fn()` unless this exact spec is already on disk. `fn` returns the
    payload dict; anything not JSON-serialisable must be checkpointed separately."""
    if store.has(spec):
        return store.get(spec)
    t0 = time.time()
    payload = fn()
    payload["_elapsed"] = time.time() - t0
    r = store.put(spec, payload)
    if label:
        print(f"  [{store.block}] {label}  {payload['_elapsed']/60:.1f}m")
    return r


def ckpt_path(name):
    return CKPTDIR / f"{name}__{TAG}.pt"


def ckpt_save(name, obj):
    p = ckpt_path(name)
    torch.save(obj, str(p) + ".tmp")
    os.replace(str(p) + ".tmp", str(p))


def ckpt_load(name):
    p = ckpt_path(name)
    if not (RESUME and p.exists()):
        return None
    try:
        return torch.load(str(p), map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(str(p), map_location=DEVICE)
    except Exception as e:
        print(f"  checkpoint {p.name} unreadable ({e}) — recomputing")
        return None


def save_model(name, m):
    ckpt_save(name, {"state": m.state_dict()})


def load_model_into(name, m):
    ck = ckpt_load(name)
    if ck is None:
        return False
    m.load_state_dict(ck["state"])
    return True


print(f"persistence: results -> {RESDIR}   checkpoints -> {CKPTDIR}")


## Task and ablation


In [ ]:
PAD, DELAY = 0, 1
N_SPECIAL = 2
MAX_POS = 24


@dataclass
class Config:
    n_symbols: int = 8
    max_query_pos: int = MAX_POS
    base_delay: int = 3
    d_model: int = 64
    lr: float = 3e-3
    batch_size: int = 128
    grad_clip: float = 1.0


CFG = Config()
N_CLASSES = CFG.n_symbols
VOCAB = N_SPECIAL + CFG.n_symbols + CFG.max_query_pos


class SelectiveRecall:
    """`x1 ... xk . . . Qj` -> report `xj`. The query is unknown until the end,
    so nothing can be stored selectively and both algorithms are viable."""
    name, n_out = "selective_recall", N_CLASSES

    def __init__(self, cfg=None):
        self.cfg = cfg or CFG

    def generate(self, B, load=LOAD, extra_delay=0, device=DEVICE):
        c = self.cfg
        lab = torch.randint(0, c.n_symbols, (B, load), device=device)
        j = torch.randint(0, load, (B,), device=device)
        ar = torch.arange(B, device=device)
        delay = torch.full((B, c.base_delay + extra_delay), DELAY,
                           device=device, dtype=torch.long)
        q = (j + N_SPECIAL + c.n_symbols).unsqueeze(1)
        return {"x": torch.cat([lab + N_SPECIAL, delay, q], 1), "y": lab[ar, j]}


def make_task():
    return SelectiveRecall()


def ablate(v, spec):
    """Resample ablation: a trial's activation is replaced with another trial's,
    so the readout still sees an in-distribution vector that is uninformative
    about this trial. Zeroing would be off-manifold — and, as this notebook
    shows, off-manifold ablation is one of the things that makes a readout
    unidentifiable."""
    if v is None or spec is None:
        return v
    mode, strength = spec if isinstance(spec, (tuple, list)) else (spec, 1.0)
    if strength <= 0:
        return v
    repl = {"shuffle": lambda: v[torch.randperm(v.shape[0], device=v.device)],
            "mean": lambda: v.mean(0, keepdim=True).expand_as(v),
            "zero": lambda: torch.zeros_like(v)}[mode]()
    return (1.0 - strength) * v + strength * repl


## Five readout structures

Same encoder, same budget, same seeds, capacities matched against an
attention-free GRU. Only the combination rule differs — and, in one case,
whether the final timestep is visible to the keys and values.


In [ ]:
# Five ways of combining a recurrent state with a retrieval read, on an
# identical GRU encoder. The question is not which one is more accurate — they
# are all at ceiling — but which one lets you tell, afterwards, which algorithm
# the network used.
#
#   additive        logits = W_h f(h_T) + W_c g(c);  keys/values EXCLUDE h_T
#   additive_incl   the same, but keys/values INCLUDE the final timestep
#   shared_proj     logits = W( f(h_T) + g(c) )      — additive in features
#   concat_mlp      logits = W( MLP([f(h_T); g(c)]) ) — the standard hybrid
#   gated           logits = W( s*f(h_T) + (1-s)*g(c) ),  s = sigma(V[h_T; c])
#
# `additive` is the design used in the main paper. Excluding the final timestep
# from the keys and values is not cosmetic: with h_T in the values, the
# retrieval route can carry a copy of the recurrent state, and ablating it then
# measures recurrence.

READOUTS = ["additive", "additive_incl", "shared_proj", "concat_mlp", "gated"]
INCLUDES_HT = {"additive": False, "additive_incl": True, "shared_proj": False,
               "concat_mlp": False, "gated": False}


class Routed(nn.Module):
    def __init__(self, readout="additive", d=None, n_out=N_CLASSES, cfg=None,
                 attention=True, recurrence=True):
        super().__init__()
        cfg = cfg or CFG
        d = d or cfg.d_model
        self.readout, self.d = readout, d
        self.attention, self.recurrence = attention, recurrence
        self.emb = nn.Embedding(VOCAB, d)
        self.rnn = nn.GRU(d, d, batch_first=True)
        self.proj_h = nn.Sequential(nn.Linear(d, d), nn.GELU())
        if attention:
            self.q, self.k, self.v = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
            self.scale = d ** -0.5
            self.proj_c = nn.Sequential(nn.Linear(d, d), nn.GELU())
        if not attention:
            self.w = nn.Linear(d, n_out)
        elif readout in ("additive", "additive_incl"):
            self.w_h, self.w_c = nn.Linear(d, n_out), nn.Linear(d, n_out, bias=False)
        elif readout == "shared_proj":
            self.w = nn.Linear(d, n_out)
        elif readout == "concat_mlp":
            self.mix = nn.Sequential(nn.Linear(2 * d, d), nn.GELU())
            self.w = nn.Linear(d, n_out)
        elif readout == "gated":
            self.gate = nn.Linear(2 * d, 1)
            self.w = nn.Linear(d, n_out)
        else:
            raise ValueError(readout)
        self.n_out = n_out

    def forward(self, x, intervene=None, null=None, drop=None):
        iv, null, drop = intervene or {}, null or {}, drop or {}
        H, _ = self.rnn(self.emb(x))
        h_T = H[:, -1, :]
        h_in = h_T
        if null.get("hidden"):
            h_in = h_in[torch.randperm(h_in.shape[0], device=h_in.device)]
        if drop.get("hidden"):
            h_in = torch.zeros_like(h_in)     # branch dropout: zeroed, not resampled,
                                              # so robustness to the test-time
                                              # perturbation is not trained for directly
        h_r = ablate(h_in, iv.get("hidden"))
        fh = self.proj_h(h_r)
        if not self.attention:
            return {"logits": self.w(fh), "feat": {"hidden": fh, "context": None},
                    "share": 0.0}

        Hk = H if INCLUDES_HT[self.readout] else H[:, :-1, :]
        w = ((self.q(h_T).unsqueeze(1) @ self.k(Hk).transpose(1, 2)).squeeze(1)
             * self.scale).softmax(-1)
        V = self.v(Hk)
        if INCLUDES_HT[self.readout]:
            # split the read into the final-timestep slot and everything else,
            # so each can be ablated on its own
            c_slot = w[:, -1:] * V[:, -1, :]
            c_rest = torch.einsum("bt,btd->bd", w[:, :-1], V[:, :-1, :])
        else:
            c_slot = torch.zeros_like(V[:, 0, :])
            c_rest = torch.einsum("bt,btd->bd", w, V)
        c_slot = ablate(c_slot, iv.get("context_slot"))
        c_rest = ablate(c_rest, iv.get("context_rest"))
        ctx = c_slot + c_rest
        if null.get("context"):
            ctx = ctx[torch.randperm(ctx.shape[0], device=ctx.device)]
        if drop.get("context"):
            ctx = torch.zeros_like(ctx)
        ctx = ablate(ctx, iv.get("context"))
        gc = self.proj_c(ctx)

        logits = self.combine(fh, gc, h_r, ctx)
        share = (gc.std(-1).mean() / (fh.std(-1).mean() + gc.std(-1).mean() + 1e-9))
        return {"logits": logits,
                "feat": {"hidden": fh, "context": gc,
                         "context_slot": c_slot, "context_rest": c_rest},
                "share": float(share.detach()), "attn_w": w,
                "slot_w": float(w[:, -1].mean().detach()) if INCLUDES_HT[self.readout] else 0.0}

    def combine(self, fh, gc, h_r, ctx):
        """The readout, isolated. Feeding it mismatched (fh, gc) is what makes
        additive separability directly testable."""
        if self.readout in ("additive", "additive_incl"):
            return self.w_h(fh) + self.w_c(gc)
        if self.readout == "shared_proj":
            return self.w(fh + gc)
        if self.readout == "concat_mlp":
            return self.w(self.mix(torch.cat([fh, gc], -1)))
        s = torch.sigmoid(self.gate(torch.cat([h_r, ctx], -1)))
        return self.w(s * fh + (1 - s) * gc)


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


def match_capacity(verbose=True):
    """Reference is the attention-free GRU, so every readout pays for its own
    extra parameters instead of being handed them."""
    target = count_params(Routed("additive", d=CFG.d_model, attention=False))
    dims, rows = {}, [{"model": "gru (reference)", "d": CFG.d_model,
                       "params": target}]
    for r in READOUTS:
        best = min(range(8, 200, 2),
                   key=lambda d: abs(count_params(Routed(r, d=d)) - target))
        dims[r] = best
        rows.append({"model": r, "d": best, "params": count_params(Routed(r, d=best))})
    dims["gru"] = CFG.d_model
    if verbose:
        T = pd.DataFrame(rows)
        T["ratio"] = (T.params / target).round(3)
        print(f"capacity matched to {target:,} parameters")
        print(T.to_string(index=False))
        T.to_csv(csv("a0_capacity"), index=False)
    return dims


DIMS = match_capacity()


def build(readout="additive", attention=True):
    if readout == "gru":
        return Routed("additive", d=DIMS["gru"], attention=False)
    return Routed(readout, d=DIMS[readout], attention=attention)


## Training, measurement, and the ground-truth probe

The price is applied to the context **feature** $g(c)$ rather than to its logit
contribution, because `concat_mlp` and `gated` have no separate logit
contribution. On the additive readout the two are equivalent up to the norm of
$W_c$, and this keeps all five comparable. Note the difference from the main
paper, where the price is on the logit contribution.

The probe is the part that makes this a calibration rather than a comparison:
it freezes the encoder and trains a fresh head on one branch alone, so it says
whether the answer is *present* in that branch without any reference to the
readout structure being tested.


In [ ]:
def train(readout, seed, steps=STEPS, lam=0.0, null=None, log_every=None,
          attention=True, drop_p=0.0):
    """The price is applied to the context FEATURE g(c) rather than to its logit
    contribution, because `concat_mlp` and `gated` have no separate logit
    contribution. On the additive readout the two are equivalent up to the
    norm of W_c, and this keeps all five structures comparable."""
    task = make_task()
    set_seed(seed)
    m = build(readout, attention=attention).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=CFG.lr)
    trace = []
    for step in range(1, steps + 1):
        m.train()
        b = task.generate(CFG.batch_size)
        dr = None
        if drop_p and random.random() < drop_p:
            # one route, chosen uniformly, is dropped this step
            dr = {"hidden": True} if random.random() < 0.5 else {"context": True}
        o = m(b["x"], null=null, drop=dr)
        loss = F.cross_entropy(o["logits"], b["y"])
        if lam > 0 and o["feat"]["context"] is not None:
            loss = loss + lam * o["feat"]["context"].norm(dim=-1).mean()
        opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(m.parameters(), CFG.grad_clip); opt.step()
        if log_every and (step % log_every == 0 or step == 1):
            trace.append({"step": step, **measure(m, task, null=null)})
    return m, task, trace


@torch.no_grad()
def accuracy(m, task, intervene=None, null=None, n_batches=8, drop=None):
    m.eval(); c = t = 0
    for _ in range(n_batches):
        b = task.generate(CFG.batch_size)
        c += (m(b["x"], intervene=intervene, null=null, drop=drop)["logits"].argmax(-1)
              == b["y"]).sum().item()
        t += b["y"].numel()
    m.train(); return c / t


@torch.no_grad()
def measure(m, task, mode="shuffle", n_batches=8, null=None):
    """The standard protocol, applied identically to every readout structure."""
    acc = accuracy(m, task, null=null, n_batches=n_batches)
    den = max(acc - 1.0 / m.n_out, 1e-6)
    d = lambda a: float(np.clip((acc - a) / den, 0, 1))
    out = {"acc_clean": acc,
           "hidden_dep": d(accuracy(m, task, {"hidden": (mode, 1.0)}, null, n_batches))}
    if not m.attention:
        out.update({"attn_dep": 0.0, "slot_dep": 0.0, "rest_dep": 0.0, "share": 0.0})
        return out
    out["attn_dep"] = d(accuracy(m, task, {"context": (mode, 1.0)}, null, n_batches))
    out["slot_dep"] = d(accuracy(m, task, {"context_slot": (mode, 1.0)}, null, n_batches))
    out["rest_dep"] = d(accuracy(m, task, {"context_rest": (mode, 1.0)}, null, n_batches))
    m.eval(); sh = []
    for _ in range(4):
        sh.append(m(task.generate(CFG.batch_size)["x"], null=null)["share"])
    m.train()
    out["share"] = float(np.mean(sh))
    return out


def probe(m, task, branch, steps=PROBE_STEPS, seed=0):
    """Decodability of the answer from one branch alone, with the encoder frozen
    and a fresh readout trained on top. This is an information-level ground
    truth that does not depend on the readout structure being tested, which is
    exactly what a calibration needs."""
    set_seed(1000 + seed)
    head = nn.Sequential(nn.Linear(m.d, m.d), nn.GELU(),
                         nn.Linear(m.d, m.n_out)).to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=3e-3)
    for p in m.parameters():
        p.requires_grad_(False)
    m.eval()
    for _ in range(steps):
        b = task.generate(CFG.batch_size)
        with torch.no_grad():
            f = m(b["x"])["feat"][branch]
        if f is None:
            return np.nan
        loss = F.cross_entropy(head(f), b["y"])
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    c = t = 0
    with torch.no_grad():
        for _ in range(8):
            b = task.generate(CFG.batch_size)
            f = m(b["x"])["feat"][branch]
            c += (head(f).argmax(-1) == b["y"]).sum().item(); t += b["y"].numel()
    for p in m.parameters():
        p.requires_grad_(True)
    m.train()
    return c / t


@torch.no_grad()
def separability(m, task, n_batches=8):
    """Independent route ablation requires the logits to be additively
    separable: L(h, c) = A(h) + B(c). If they are, ablating one route leaves the
    other route's contribution untouched, which is the whole basis for reading
    an ablation as evidence about one algorithm.

    Separability is testable exactly, with no training and no ground-truth
    construction, by the mixed second difference

        DD = L(h,c) - L(h,c') - L(h',c) + L(h',c')

    which is identically zero for a separable readout and positive otherwise.
    Reported relative to the effect of swapping h alone, so it is a fraction of
    a real effect rather than an arbitrary scale.
    """
    if not m.attention:
        return 0.0
    m.eval(); num, den = [], []
    for _ in range(n_batches):
        b = task.generate(CFG.batch_size)
        H, _ = m.rnn(m.emb(b["x"]))
        h_T = H[:, -1, :]
        Hk = H if INCLUDES_HT[m.readout] else H[:, :-1, :]
        w = ((m.q(h_T).unsqueeze(1) @ m.k(Hk).transpose(1, 2)).squeeze(1)
             * m.scale).softmax(-1)
        ctx = torch.einsum("bt,btd->bd", w, m.v(Hk))
        fh, gc = m.proj_h(h_T), m.proj_c(ctx)
        pi = torch.randperm(fh.shape[0], device=fh.device)
        L11 = m.combine(fh, gc, h_T, ctx)
        L12 = m.combine(fh, gc[pi], h_T, ctx[pi])
        L21 = m.combine(fh[pi], gc, h_T[pi], ctx)
        L22 = m.combine(fh[pi], gc[pi], h_T[pi], ctx[pi])
        num.append((L11 - L12 - L21 + L22).norm(dim=-1))
        den.append((L11 - L21).norm(dim=-1))
    m.train()
    return float(torch.cat(num).mean() / (torch.cat(den).mean() + 1e-9))


@torch.no_grad()
def slot_weight(m, task, n_batches=8):
    """Attention mass on the final timestep. Unlike an ablation-derived number
    this does not saturate when one route dominates, so it still has range in
    the regime where the deps have pinned at 1."""
    if not m.attention or not INCLUDES_HT[m.readout]:
        return 0.0
    m.eval()
    v = [m(task.generate(CFG.batch_size)["x"])["slot_w"] for _ in range(n_batches)]
    m.train()
    return float(np.mean(v))


def pm(s):
    v = np.asarray([x for x in s if x == x], float)
    if not len(v):
        return "--"
    return f"{v.mean():.3f} ± {v.std():.3f}" if len(v) > 1 else f"{v.mean():.3f}"


## Does the inversion survive a non-separable readout?

The additive readout is the identification instrument, which invites the
objection that it manufactures the finding. The frozen-probe deltas reported
earlier are a weaker answer than the main analysis, so this section runs the
actual dependence machinery on readouts that are *not* additively separable.

The trick is where the intervention is applied. Resampling a route's **logit
contribution** requires the logits to decompose by route. Resampling its
**pre-fusion activation** — f(h_T) or g(c), before they meet — is defined for
any fusion rule. Same intervention, one layer earlier.

Three readouts, chosen by their measured separability error: `additive` (0.00,
exactly separable), `concat_mlp` (0.55) and `gated` (1.71). If the inversion
appears in all three, the additive form is not creating it; it is what makes it
legible. Their dependence numbers will be noisier, and that noise is the
argument for the instrument rather than a hedge against it.


In [ ]:
# Does the route inversion appear in readouts that are NOT additively separable?
#
# The additive readout is this paper's identification instrument, so the
# standing objection is that it manufactures the finding. The existing answer —
# frozen-probe deltas — is weaker than the main analysis and has been flagged as
# such repeatedly. This runs the actual dependence machinery on non-separable
# readouts by resampling the PRE-FUSION route activations, f(h_T) and g(c),
# rather than their logit contributions. It is the same intervention applied one
# layer earlier, where it is defined regardless of how the routes are combined.
#
# The price is on the context FEATURE g(c) rather than its logit contribution,
# because concat_mlp and gated have no separate logit contribution. On the
# additive readout the two are equivalent up to the norm of W_c; this keeps all
# three readouts under the same penalty. Say so in the paper.
READOUTS_T = ["additive", "concat_mlp", "gated"]
LAMS_T = [0.0, 0.10]

st = Store("prefusion")
for r, lam, sd in itertools.product(READOUTS_T, LAMS_T, SEEDS):
    spec = {"block": "prefusion", "readout": r, "lam": lam, "seed": sd,
            "steps": STEPS}

    def _go(r=r, lam=lam, sd=sd):
        m, task, _ = train(r, sd, lam=lam)
        acc = accuracy(m, task)
        den = max(acc - 1.0 / m.n_out, 1e-6)
        d = lambda a: float(np.clip((acc - a) / den, 0, 1))
        return {"acc_clean": acc,
                "hidden_dep": d(accuracy(m, task, {"hidden": ("shuffle", 1.0)})),
                "attn_dep": d(accuracy(m, task, {"context": ("shuffle", 1.0)})),
                "probe_hidden": probe(m, task, "hidden", seed=sd),
                "probe_context": probe(m, task, "context", seed=sd),
                "separability": separability(m, task),
                "share": float(np.mean([
                    m(task.generate(CFG.batch_size)["x"])["share"]
                    for _ in range(4)]))}

    cached(st, spec, _go, label=f"{r:<12} lam={lam} seed {sd}")

PF = st.frame(); PF.to_csv(csv("p20_prefusion"), index=False)
print("\npre-fusion route dependence, by readout and price:\n")
print(PF.groupby(["readout", "lam"])[
    ["acc_clean", "share", "hidden_dep", "attn_dep", "probe_hidden",
     "separability"]].mean().round(3).to_string())

print("\nchange under price (lambda 0 -> 0.10), mean +/- sd over seeds:\n")
rows = []
for r in READOUTS_T:
    a = PF[(PF.readout == r) & (PF.lam == 0.0)]
    b = PF[(PF.readout == r) & (PF.lam == LAMS_T[-1])]
    if not len(a) or not len(b):
        continue
    inv = b.hidden_dep.mean() > b.attn_dep.mean()
    n_inv = int((b.hidden_dep.values > b.attn_dep.values).sum())
    rows.append({"readout": r, "sep_error": b.separability.mean(),
                 "hidden_0": a.hidden_dep.mean(), "hidden_1": b.hidden_dep.mean(),
                 "hidden_1_sd": b.hidden_dep.std(),
                 "attn_0": a.attn_dep.mean(), "attn_1": b.attn_dep.mean(),
                 "attn_1_sd": b.attn_dep.std(),
                 "share_0": a.share.mean(), "share_1": b.share.mean(),
                 "acc_1": b.acc_clean.mean(), "inverted": inv,
                 "seeds_inverted": n_inv, "n_seeds": len(b)})
    print(f"  {r:<12} hidden_dep {a.hidden_dep.mean():.3f} -> "
          f"{b.hidden_dep.mean():.3f}+/-{b.hidden_dep.std():.3f}"
          f"   attn_dep {a.attn_dep.mean():.3f} -> "
          f"{b.attn_dep.mean():.3f}+/-{b.attn_dep.std():.3f}")
    print(f"  {'':<12} share {a.share.mean():.3f} -> {b.share.mean():.3f}"
          f"   acc {b.acc_clean.mean():.4f}"
          f"   separability error {b.separability.mean():.2f}"
          f"   {n_inv}/{len(b)} seeds inverted"
          f"{'   INVERTS' if inv else ''}")
PFS = pd.DataFrame(rows)
PFS.to_csv(csv("p20_prefusion_summary"), index=False)

if len(PFS):
    nonsep = PFS[PFS.sep_error > 0.05]
    print("\n" + "=" * 74)
    if len(nonsep) and nonsep.inverted.all():
        print("  The inversion appears in readouts whose cross-terms are large")
        print(f"  ({', '.join(f'{r.readout} {r.sep_error:.2f}' for _, r in nonsep.iterrows())}).")
        print("  The additive readout is therefore not creating the phenomenon —")
        print("  it is what lets the phenomenon be measured cleanly. That is the")
        print("  argument for the instrument, and it is now a measurement rather")
        print("  than an assertion.")
    elif len(nonsep):
        print("  The inversion does NOT reproduce in every non-separable readout.")
        print("  Report which ones and their separability errors. A dependence")
        print("  measure applied through a fusion with a large cross-term is")
        print("  measuring the fusion as well as the route, so a null there is")
        print("  ambiguous — say that rather than claiming architecture-generality.")
    print("\n  Report the separability error beside every dependence number: 0.00")
    print("  for additive, and whatever concat_mlp and gated give. Those readouts")
    print("  have real cross-terms, so their numbers are noisier by construction,")
    print("  and the noise is the reason the instrument exists.")
    print("=" * 74)

# The information-level check, which needs no readout assumption at all
print("\ndecodability of the answer from the recurrent state (frozen encoder,")
print("fresh probe — independent of how the routes are fused):")
print(PF.pivot_table(index="readout", columns="lam",
                     values="probe_hidden").round(3).to_string())
print("\n  A rise here under price says the answer became available in the")
print("  recurrent state, whatever the readout does with it. It is the weakest")
print("  claim of the three and the one that depends on nothing.")

import shutil
shutil.make_archive("/kaggle/working/results_backup", "zip", RESDIR)
print("\nbacked up")
